# Tenant-Wide Semantic Model Incremental Refresh Audit (v2 - XMLA only)

This notebook finds **which semantic models are configured with incremental refresh and which are not**, by reading each model's **table-level `RefreshPolicy`** over the **XMLA endpoint** (via `semantic-link-labs` / TOM).

## Why XMLA
Incremental refresh is a **table-level refresh policy** driven by the reserved `RangeStart` / `RangeEnd` Power Query parameters. The service creates and manages the partitions, and the policy itself is only exposed through the **XMLA endpoint** - no Power BI REST API returns it.

## Requirements and limitations
- Runs **inside a Microsoft Fabric notebook** (or any environment with XMLA connectivity).
- The identity needs **Fabric / Power BI Admin** rights to list all datasets, plus access to each workspace to read its model.
- The model must be on a capacity that exposes the **XMLA read endpoint** - **Premium / PPU / Fabric**. Models on shared **Pro** capacity have **no XMLA endpoint** and are reported as `Unknown`.
- **Dataflows are not covered** - XMLA applies to semantic models only.
- Requires the `semantic-link-labs` package (installed in section 1).

## 0. Install dependencies

`semantic-link-labs` (`sempy_labs`) provides the TOM wrapper used to read each model's `RefreshPolicy` over XMLA. Run once per session.

In [ ]:
# Install semantic-link-labs (provides the TOM wrapper for XMLA reads).
# Run this ONCE per session. If the import check below fails, RESTART the
# kernel and run the notebook again from the top.
%pip install --upgrade pip
%pip install --upgrade "pyjwt>=2.6.0"
%pip install semantic-link-labs
try:
    import sempy_labs
    from sempy_labs.tom import connect_semantic_model  # noqa: F401
    print('OK - sempy_labs is importable (version:', getattr(sempy_labs, '__version__', 'unknown'), ')')
except ImportError as e:
    print('sempy_labs NOT importable yet. RESTART THE KERNEL, then re-run from the top.')
    print('Details:', e)

## 1. Parameters

This cell is tagged **parameters** so it can be overridden from a Fabric pipeline or scheduled job.

In [ ]:
# Authentication for the Admin listing API:
# 'fabric_integrated' (default, notebook identity), 'service_principal', or 'default_azure'
auth_method = 'fabric_integrated'

# Only required when auth_method == 'service_principal'
tenant_id = ''
client_id = ''
client_secret = ''

# Parallel XMLA/TOM connections. Keep low to avoid overloading the XMLA endpoints.
max_workers = 4

# Only probe workspaces on dedicated (Premium/PPU/Fabric) capacity. Datasets on
# shared (Pro) capacity have no XMLA endpoint and are reported as 'Unknown'.
skip_non_capacity_workspaces = True

# Optionally restrict the scan to specific workspace IDs (empty list = whole tenant).
workspace_ids_filter = []

# Save results to an attached Lakehouse as a Delta table
save_to_lakehouse = False
lakehouse_table_name = 'semantic_model_incremental_refresh'

## 2. Imports and logging

In [ ]:
import os
import time
import logging
from typing import Dict, List, Optional
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 3. Authentication

Used only to **list** workspaces and datasets via the Admin API. The XMLA connections in section 5 authenticate through the notebook identity.

In [ ]:
class FabricAuthenticator:
    """Gets an access token for the Power BI Admin API."""

    PBI_RESOURCE = 'https://analysis.windows.net/powerbi/api'
    PBI_SCOPE = 'https://analysis.windows.net/powerbi/api/.default'

    def __init__(self, tenant_id=None, client_id=None, client_secret=None, auth_method='fabric_integrated'):
        self.tenant_id = tenant_id or os.getenv('AZURE_TENANT_ID')
        self.client_id = client_id or os.getenv('AZURE_CLIENT_ID')
        self.client_secret = client_secret or os.getenv('AZURE_CLIENT_SECRET')
        self.auth_method = auth_method

    def get_access_token(self):
        if self.auth_method == 'fabric_integrated':
            import notebookutils  # available in the Fabric runtime
            logger.info('Authenticated using Fabric notebook identity')
            return notebookutils.credentials.getToken(self.PBI_RESOURCE)
        if self.auth_method == 'service_principal':
            from azure.identity import ClientSecretCredential
            if not all([self.tenant_id, self.client_id, self.client_secret]):
                raise ValueError('service_principal auth requires tenant_id, client_id and client_secret')
            cred = ClientSecretCredential(tenant_id=self.tenant_id, client_id=self.client_id, client_secret=self.client_secret)
            logger.info('Authenticated using service principal')
            return cred.get_token(self.PBI_SCOPE).token
        from azure.identity import DefaultAzureCredential
        cred = DefaultAzureCredential()
        logger.info('Authenticated using DefaultAzureCredential')
        return cred.get_token(self.PBI_SCOPE).token

## 4. Admin API client

Lists all workspaces (`/admin/groups`) and semantic models (`/admin/datasets`) so we know what to probe. The `isOnDedicatedCapacity` flag lets us skip Pro workspaces that have no XMLA endpoint.

In [ ]:
class PowerBIAdminAPIClient:
    """Minimal Power BI Admin API client (list workspaces and datasets)."""

    ADMIN_BASE_URL = 'https://api.powerbi.com/v1.0/myorg/admin'

    def __init__(self, authenticator):
        self.authenticator = authenticator
        self.session = requests.Session()
        token = self.authenticator.get_access_token()
        self.headers = {'Authorization': 'Bearer ' + token, 'Content-Type': 'application/json'}

    def _get(self, endpoint, params=None, retry_count=3):
        url = self.ADMIN_BASE_URL + endpoint
        for attempt in range(retry_count):
            try:
                r = self.session.get(url, headers=self.headers, params=params, timeout=30)
                if r.status_code == 429:
                    wait = int(r.headers.get('Retry-After', 60))
                    logger.warning('Rate limited; waiting %ss...', wait)
                    time.sleep(wait)
                    continue
                r.raise_for_status()
                return r.json() if r.text else {}
            except requests.exceptions.RequestException as e:
                if attempt < retry_count - 1:
                    time.sleep(2 ** attempt)
                else:
                    logger.error('Request to %s failed: %s', endpoint, e)
                    raise
        return {}

    def _paged(self, endpoint):
        items, skip, top = [], 0, 100
        while True:
            resp = self._get(endpoint, params={'$skip': skip, '$top': top})
            batch = resp.get('value', [])
            if not batch:
                break
            items.extend(batch)
            skip += top
        return items

    def get_workspaces(self):
        logger.info('Fetching workspaces via /admin/groups...')
        ws = self._paged('/groups')
        logger.info('Workspaces found: %d', len(ws))
        return ws

    def get_datasets(self):
        logger.info('Fetching datasets via /admin/datasets...')
        ds = self._paged('/datasets')
        logger.info('Datasets found: %d', len(ds))
        return ds

## 5. Incremental Refresh Detector (XMLA / TOM)

Connects to each model read-only and inspects every table's `RefreshPolicy`. A non-null policy means incremental refresh is configured; the store/refresh windows and mode are captured for detail.

| Field | Meaning |
|---|---|
| `is_incremental` | `True` / `False` (read via XMLA) or `Unknown` (could not read) |
| `incremental_tables` | Tables that carry a refresh policy |
| `policy_details` | Store window, refresh window, hybrid/real-time, detect-changes |
| `detection_source` | `XMLA`, `No XMLA (shared/Pro capacity)`, or `Unavailable` |

In [ ]:
class IncrementalRefreshDetector:
    """Detect incremental-refresh policies via the XMLA endpoint (TOM)."""

    def __init__(self):
        self.available = False
        self._connect = None
        try:
            from sempy_labs.tom import connect_semantic_model
            self._connect = connect_semantic_model
            self.available = True
            logger.info('semantic-link-labs available: XMLA incremental refresh detection enabled.')
        except Exception as e:
            logger.error('semantic-link-labs (sempy_labs) not importable - run the install cell first. (%s)', e)

    @staticmethod
    def _as_str(value):
        try:
            return str(value)
        except Exception:
            return ''

    def _describe_policy(self, table_name, policy):
        """Return the three policy values: table, store (archive) window, refresh window."""
        # Archive data starting before refresh date (store window).
        store = '%s %s' % (policy.RollingWindowPeriods, self._as_str(policy.RollingWindowGranularity))
        # Incrementally refresh data starting before refresh date (refresh window).
        refresh = '%s %s' % (policy.IncrementalPeriods, self._as_str(policy.IncrementalGranularity))
        return {'table': table_name, 'store_window': store, 'refresh_window': refresh}

    def detect(self, workspace_id, dataset_id):
        result = {
            'is_incremental': 'Unknown',
            'incremental_tables': [],
            'policy_details': '',
            'detection_source': 'XMLA',
            'error': '',
        }
        if not self.available:
            result['detection_source'] = 'Unavailable (sempy_labs not installed)'
            return result
        try:
            with self._connect(dataset=dataset_id, workspace=workspace_id, readonly=True) as tom:
                incr_tables, details = [], []
                for t in tom.model.Tables:
                    policy = getattr(t, 'RefreshPolicy', None)
                    if policy is None:
                        continue
                    incr_tables.append(t.Name)
                    try:
                        p = self._describe_policy(t.Name, policy)
                        details.append("%s: store %s, refresh %s" % (p['table'], p['store_window'], p['refresh_window']))
                    except Exception:
                        details.append(t.Name)
                result['is_incremental'] = len(incr_tables) > 0
                result['incremental_tables'] = incr_tables
                result['policy_details'] = ' | '.join(details)
        except Exception as e:
            result['error'] = str(e)
            logger.debug('XMLA read failed for dataset %s in workspace %s: %s', dataset_id, workspace_id, e)
        return result

## 6. Audit engine

Lists workspaces and datasets, probes each capacity-backed model over XMLA, and builds the results table. Datasets on shared capacity are marked `Unknown` without a connection attempt.

In [ ]:
class TenantIncrementalRefreshAudit:
    """Tenant-wide XMLA incremental-refresh audit for semantic models."""

    def __init__(self, client, detector):
        self.client = client
        self.detector = detector
        self.workspaces = {}
        self.datasets = []

    def run(self, max_workers=4, skip_non_capacity=True, workspace_ids_filter=None):
        start = datetime.now()
        self._fetch_workspaces()
        self._fetch_datasets(workspace_ids_filter)
        self._detect(max_workers=max_workers, skip_non_capacity=skip_non_capacity)
        df = self._build_df()
        logger.info('Audit completed in %.1fs: %d datasets across %d workspaces.',
                    (datetime.now() - start).total_seconds(), len(df), len(self.workspaces))
        return df

    def _fetch_workspaces(self):
        for ws in self.client.get_workspaces():
            self.workspaces[ws['id']] = {
                'name': ws.get('name', 'Unknown'),
                'isOnDedicatedCapacity': ws.get('isOnDedicatedCapacity', False),
                'capacityId': ws.get('capacityId', ''),
                'type': ws.get('type', 'Unknown'),
                'state': ws.get('state', 'Unknown'),
            }

    def _fetch_datasets(self, workspace_ids_filter=None):
        self.datasets = self.client.get_datasets()
        if workspace_ids_filter:
            allowed = set(workspace_ids_filter)
            self.datasets = [d for d in self.datasets if d.get('workspaceId') in allowed]
        logger.info('Datasets to evaluate: %d', len(self.datasets))

    def _detect(self, max_workers=4, skip_non_capacity=True):
        if not self.detector.available:
            logger.warning('Detector unavailable; all datasets will be reported as Unknown.')

        candidates = []
        for ds in self.datasets:
            ws = self.workspaces.get(ds.get('workspaceId'), {})
            on_capacity = ws.get('isOnDedicatedCapacity', False)
            if skip_non_capacity and not on_capacity:
                ds['_result'] = {
                    'is_incremental': 'Unknown',
                    'incremental_tables': [],
                    'policy_details': '',
                    'detection_source': 'No XMLA (shared/Pro capacity)',
                    'error': '',
                }
            else:
                candidates.append(ds)
        logger.info('XMLA probing %d datasets (skipped %d on shared capacity)...',
                    len(candidates), len(self.datasets) - len(candidates))

        def _probe(ds):
            return self.detector.detect(ds.get('workspaceId'), ds['id'])

        with ThreadPoolExecutor(max_workers=max_workers) as ex:
            futures = {ex.submit(_probe, ds): ds for ds in candidates}
            done = 0
            for fut in as_completed(futures):
                try:
                    futures[fut]['_result'] = fut.result()
                except Exception as e:
                    futures[fut]['_result'] = {
                        'is_incremental': 'Unknown', 'incremental_tables': [],
                        'policy_details': '', 'detection_source': 'XMLA', 'error': str(e),
                    }
                done += 1
                if done % 50 == 0:
                    logger.info('Probed %d/%d datasets', done, len(candidates))

    def _build_df(self):
        rows = []
        for ds in self.datasets:
            ws = self.workspaces.get(ds.get('workspaceId', ''), {})
            res = ds.get('_result', {})
            rows.append({
                'Workspace Name': ws.get('name', 'Unknown'),
                'Workspace ID': ds.get('workspaceId', ''),
                'On Dedicated Capacity': ws.get('isOnDedicatedCapacity', False),
                'Dataset Name': ds.get('name', 'Unknown'),
                'Dataset ID': ds.get('id', ''),
                'Is Incremental': res.get('is_incremental', 'Unknown'),
                'Incremental Tables': ', '.join(res.get('incremental_tables', [])) or 'None',
                'Policy Details': res.get('policy_details', '') or 'None',
                'Detection Source': res.get('detection_source', 'Unknown'),
                'Is Refreshable': ds.get('isRefreshable', 'Unknown'),
                'Target Storage Mode': ds.get('targetStorageMode', 'Unknown'),
                'Owner': ds.get('configuredBy', 'Unknown'),
                'Error': res.get('error', ''),
            })
        df = pd.DataFrame(rows)
        if not df.empty:
            df = df.sort_values(['Workspace Name', 'Dataset Name']).reset_index(drop=True)
        return df

## 7. Run the audit

In [ ]:
authenticator = FabricAuthenticator(
    tenant_id=tenant_id or None,
    client_id=client_id or None,
    client_secret=client_secret or None,
    auth_method=auth_method,
)
client = PowerBIAdminAPIClient(authenticator)
detector = IncrementalRefreshDetector()

audit = TenantIncrementalRefreshAudit(client, detector)
results_df = audit.run(
    max_workers=max_workers,
    skip_non_capacity=skip_non_capacity_workspaces,
    workspace_ids_filter=workspace_ids_filter or None,
)

print('Total semantic models:', len(results_df))
if not results_df.empty:
    print('Incremental Refresh Distribution:')
    print(results_df['Is Incremental'].value_counts(dropna=False))
    print('Detection Source Distribution:')
    print(results_df['Detection Source'].value_counts(dropna=False))
    print('Models WITH incremental refresh:', int((results_df['Is Incremental'] == True).sum()))
    print('Models WITHOUT incremental refresh:', int((results_df['Is Incremental'] == False).sum()))

In [ ]:
# Full results
display(results_df)

## 8. Explore results

In [ ]:
# Models configured WITH incremental refresh
display(results_df[results_df['Is Incremental'] == True])

In [ ]:
# Models WITHOUT incremental refresh (definitively read via XMLA)
display(results_df[results_df['Is Incremental'] == False])

In [ ]:
# Models that could not be read over XMLA (e.g. shared/Pro capacity, no access)
display(results_df[results_df['Is Incremental'] == 'Unknown'])

## 9. Save to Lakehouse (optional)

Set `save_to_lakehouse = True` in the parameters cell and attach a Lakehouse.

In [ ]:
if save_to_lakehouse and not results_df.empty:
    export_df = results_df.copy()
    export_df['Audit Timestamp'] = datetime.utcnow().isoformat()
    export_df.columns = [c.replace(' ', '_') for c in export_df.columns]
    spark_df = spark.createDataFrame(export_df)
    (spark_df.write
        .format('delta')
        .mode('overwrite')
        .option('mergeSchema', 'true')
        .saveAsTable(lakehouse_table_name))
    print('Saved to Lakehouse table:', lakehouse_table_name)
else:
    print('Skipping Lakehouse save (disabled or no results).')